# Non-Destructive Corpus Preprocessing for Information Retrieval

## Overview

In Notebook 01, we identified recurring scraper artifacts in the raw CNN dataset—such as trailing `"- CNN"` headline suffixes, `"(CNN)"` dateline prefixes, embedded URLs, contributor footers, and newsletter subscription promotions—along with 33 articles that contain no usable body text.

In this notebook, we prepare a clean, retrieval-ready dataset (`cnn_english_news_clean.parquet`) while adhering to four engineering principles:

1. **Preserve the Raw Corpus**: We never overwrite `cnn_english_news.parquet` and we retain all original columns bit-identically alongside the newly created clean fields.
2. **High-Precision Boilerplate Removal**: We remove only verified editorial boilerplate (sign-up banners, contributor credits, dateline markers) while keeping substantive text—including `"Story highlights"` summaries—intact.
3. **Flag Rather Than Delete**: Instead of silently dropping problematic records, we attach explicit boolean quality flags (`flag_empty_article`, `flag_dup_headline`, `flag_no_date`) so downstream retrievers can filter transparently.
4. **Separate Text Normalization from Ranking Weights**: We provide clean fields (`headline_clean`, `description_clean`, `article_clean`, `search_text`) without baking arbitrary field repetitions into the stored text; field weighting belongs in the retrieval model itself.

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 90)

RAW_PATH = Path("../data/cnn_english_news.parquet")
OUT_PATH = Path("../data/cnn_english_news_clean.parquet")

df = pd.read_parquet(RAW_PATH)
assert len(df) == 37949, "Unexpected row count in raw dataset!"
print(f"Loaded raw dataset: {len(df):,} articles x {df.shape[1]} columns")

Loaded raw dataset: 37,949 articles x 11 columns


## 1. Auditing Boilerplate Patterns

Before stripping any text from the corpus, we define targeted regular expressions for the five boilerplate patterns discovered during exploratory analysis and count how many articles each pattern matches.

In [2]:
BOILERPLATE = {
    "contributed_to_report": re.compile(
        r"CNN(?:\.com)?(?:'s[^.]{0,120}?)?\s+contributed to this report\.?"
    ),
    "signup_ad": re.compile(r"Sign up (?:for|to)[^.]{0,300}\."),
    "version_of_story": re.compile(r"A version of this story[^.]{0,200}\."),
    "join_conversation": re.compile(r"Join the conversation.*$"),
    "follow_cnn": re.compile(r"Follow CNN[^.]{0,100}"),
}

text = df["Article text"].fillna("")
for name, pat in BOILERPLATE.items():
    hits = text.str.contains(pat)
    print(f"{name:22s} {hits.sum():6d} articles ({hits.mean()*100:4.1f}%)")

contributed_to_report    4785 articles (12.6%)
signup_ad                 905 articles ( 2.4%)
version_of_story          204 articles ( 0.5%)


join_conversation          59 articles ( 0.2%)
follow_cnn                 97 articles ( 0.3%)


To verify that our `signup_ad` pattern does not accidentally remove genuine news sentences containing the phrase *"sign up for"*, we inspect a sample of matched substrings.

In [3]:
from src.tokenizer import slice_by_tokens

hits = df[text.str.contains(BOILERPLATE["signup_ad"])].copy()
hits["match"] = hits["Article text"].str.extract("(" + BOILERPLATE["signup_ad"].pattern + ")")[0]
[slice_by_tokens(m, max_tokens=28) for m in hits["match"].dropna().drop_duplicates().head(8)]

["Sign up for CNN Opinion's newsletter.", "Sign up for CNN Opinion's new newsletter.", "Sign up for the Two° newsletterFollow John Sutter on Twitter*Math behind the 70-mile meal: Here's how I calculated ...", 'Sign up for the "2 degrees" newsletter if you want to join our freewheeling book club.', "Sign up for CNN's Stress, But Less newsletter.", 'Sign up to get this weekly column as a newsletter.', "Sign up for CNN's Wonder Theory science newsletter.", 'Sign up to get weekly weather updatesHigh risk for avalanchesThe "extreme" Level 4 out of 5 atmospheric river is bringing ...']

Every sampled match is an editorial newsletter advertisement (such as *Meanwhile in China*, *5 Things*, or *Royal News*). Removing these prevents irrelevant matches when queries mention newsletters, subscriptions, or global regions like China or the UK.

## 2. Cleaning Article Bodies

We clean each article body in five deterministic steps:

1. Insert a missing separator after glued story-highlight headers (`"Story highlightsKaleb..."` $\rightarrow$ `"Story highlights: Kaleb..."`) so the first token is tokenized properly.
2. Strip the verified boilerplate patterns defined above.
3. Strip leading `"(CNN)"` dateline markers at the start of the body.
4. Remove raw `http://` and `https://` URLs embedded in the prose.
5. Collapse repeated horizontal whitespace.

In [4]:
from src.tokenizer import count_bpe_tokens_batch

LEAD_IN = re.compile(r"^\s*\(?CNN\)?(?:\s*\(?CNN\)?)*\s*")
URL = re.compile(r"https?://\S+")
WS = re.compile(r"[ \t]{2,}")

def clean_article(t: str) -> str:
    if not t:
        return ""
    # 1) Separate glued "Story highlights" prefixes
    t = re.sub(r"Story highlights(?!\s)", "Story highlights: ", t)
    # 2) Strip verified boilerplate sentences
    for pat in BOILERPLATE.values():
        t = pat.sub("", t)
    # 3) Strip leading (CNN) datelines
    t = LEAD_IN.sub("", t)
    # 4) Remove raw URLs in body text
    t = URL.sub("", t)
    # 5) Normalize horizontal whitespace
    return WS.sub(" ", t).strip()

df["article_clean"] = df["Article text"].fillna("").map(clean_article)

removed_tokens = count_bpe_tokens_batch(df["Article text"].fillna("").tolist()) - count_bpe_tokens_batch(df["article_clean"].tolist())
print(f"Average BPE tokens removed per article by boilerplate cleaning: {removed_tokens.mean():.1f} BPE tokens")
n_empty = df["article_clean"].str.strip().eq("").sum()
print(
    f"Articles with empty cleaned body: {n_empty} "
    f"(9 NaN + 17 whitespace-only + {n_empty - 26} boilerplate-only)"
)

Average BPE tokens removed per article by boilerplate cleaning: 7.2 BPE tokens
Articles with empty cleaned body: 33 (9 NaN + 17 whitespace-only + 7 boilerplate-only)


Let us compare the beginning and end of an article before and after cleaning to confirm that only the dateline prefix and contributor footer were removed.

In [5]:
from src.tokenizer import count_bpe_tokens, slice_by_tokens

i = text.str.contains(BOILERPLATE["contributed_to_report"]).idxmax()
raw, clean = df.loc[i, "Article text"], df.loc[i, "article_clean"]
print(f"Token delta  : {count_bpe_tokens(raw)} raw BPE tokens -> {count_bpe_tokens(clean)} clean BPE tokens")
print("RAW   (tail, last 35 BPE tokens) :", repr(slice_by_tokens(raw, max_tokens=35, from_end=True)))
print("CLEAN (tail, last 35 BPE tokens) :", repr(slice_by_tokens(clean, max_tokens=35, from_end=True)))
print("\nRAW   (head, first 28 BPE tokens):", repr(slice_by_tokens(raw, max_tokens=28)))
print("CLEAN (head, first 28 BPE tokens):", repr(slice_by_tokens(clean, max_tokens=28)))

Token delta  : 1941 raw BPE tokens -> 1922 clean BPE tokens
RAW   (tail, last 35 BPE tokens) : '... place for the J&J vaccine in the long-term picture of this pandemic," Montefiori said.CNN\'s Jacqueline Howard and Tasnim Ahmed contributed to this report.'
CLEAN (tail, last 35 BPE tokens) : '... the vaccine deserves more study."I think there might still be a place for the J&J vaccine in the long-term picture of this pandemic," Montefiori said.'

RAW   (head, first 28 BPE tokens): ' (CNN)The US public and even some health experts may have underestimated the Covid-19 vaccine made by Janssen, a division ...'
CLEAN (head, first 28 BPE tokens): 'The US public and even some health experts may have underestimated the Covid-19 vaccine made by Janssen, a division of Johnson & ...'


## 3. Normalizing Headlines, Dates, and URL Slugs

Scraped headlines frequently include a trailing site suffix such as `"- CNN"` or `"| CNN Business"`. We strip these suffixes to form `headline_clean`, normalize `Description` into `description_clean`, parse `Date published` into a timestamp (`published`, `year`, `month`), and extract the readable article `slug` from the URL.

In [6]:
CNN_SUFFIX = re.compile(
    r"\s*[-|–]\s*CNN(?:\s?Business|\s?News|Politics|Money|Style|Digital)?\s*$",
    re.I,
)

df["headline_clean"] = (
    df["Headline"].fillna("").str.replace(CNN_SUFFIX, "", regex=True).str.strip()
)
df["description_clean"] = df["Description"].fillna("").str.strip()

df["published"] = pd.to_datetime(df["Date published"], errors="coerce")
df["year"] = df["published"].dt.year
df["month"] = df["published"].dt.month

slug = df["Url"].str.extract(r"/([a-z0-9-]+)/[^/]*$")[0]
df["slug"] = slug
print(f"Unparsed URL slugs: {df['slug'].isna().sum()}")
df[["headline_clean", "published", "year", "slug"]].head(3)

Unparsed URL slugs: 0


,headline_clean,published,year,slug
0,"There's a shortage of truckers, but TuSimple thinks it has a solution: no driver needed",2021-07-15 02:46:59,2021,tusimple-autonomous-truck-spc-intl
1,Bioservo's robotic 'Ironhand' could protect factory workers from injuries,2021-05-12 07:52:09,2021,ironhand-exoskeleton-glove-spc-intl
2,This swarm of robots gets smarter the more it works,2021-06-16 02:51:30,2021,swarm-robots-hong-kong-warehouse-hnk-spc-intl


## 4. Non-Destructive Quality Flags

Rather than deleting rows from the table, we record data-quality anomalies as boolean columns:

- `flag_empty_article`: `True` when `article_clean` contains no usable text (33 articles in total: 9 `NaN`, 17 whitespace-only, and 7 that consisted solely of a contributor line).
- `flag_dup_headline`: `True` when `headline_clean` appears more than once in the corpus.
- `flag_no_date`: `True` when the publication timestamp could not be parsed (0 articles).

In [7]:
df["flag_empty_article"] = df["article_clean"].str.strip().eq("")
df["flag_dup_headline"] = df["headline_clean"].duplicated(keep=False)
df["flag_no_date"] = df["published"].isna()

df[["flag_empty_article", "flag_dup_headline", "flag_no_date"]].sum().to_frame("flagged_articles")

,flagged_articles
flag_empty_article,33
flag_dup_headline,175
flag_no_date,0


## 5. Assembling the Unfielded Search Document & Precomputing BPE Token Counts

For baseline single-field retrievers (such as flat Okapi BM25 in Notebook 03), we construct `search_text` by concatenating `headline_clean`, `description_clean`, and `article_clean`. Because **tokens are the universal currency of our retrieval and LLM stack**, we also precompute exact BPE subword token counts (`n_tokens_headline`, `n_tokens_description`, `n_tokens_clean`, and `n_tokens_search_text`) using the 151,643-vocabulary Qwen/Harrier BPE tokenizer (`src.tokenizer.count_bpe_tokens_batch`).

In [8]:
from src.tokenizer import count_bpe_tokens_batch, slice_by_tokens

df["search_text"] = (
    df["headline_clean"].fillna("") + " \n"
    + df["description_clean"].fillna("") + " \n"
    + df["article_clean"]
).str.strip()

df["n_tokens_headline"] = count_bpe_tokens_batch(df["headline_clean"].fillna("").tolist())
df["n_tokens_description"] = count_bpe_tokens_batch(df["description_clean"].fillna("").tolist())
df["n_tokens_clean"] = count_bpe_tokens_batch(df["article_clean"].fillna("").tolist())
df["n_tokens_search_text"] = count_bpe_tokens_batch(df["search_text"].fillna("").tolist())

print("=== Glass-Box Example: Unfielded 'search_text' (Flat BM25) vs. Separate Fielded Columns (BM25F / PassageChunker) ===")
ex = df[~df["flag_empty_article"]].iloc[0]
print(f"[1] Unfielded search_text ({ex['n_tokens_search_text']} BPE tokens | first 50 BPE tokens shown):")
print("   ", repr(slice_by_tokens(ex["search_text"], max_tokens=50)))
print("\n[2] Separate Fielded Inputs (preserved independently for BM25F and Contextual Chunking):")
print(f"  • headline_clean    ({ex['n_tokens_headline']:4d} BPE tokens): {ex['headline_clean']!r}")
print(f"  • description_clean ({ex['n_tokens_description']:4d} BPE tokens): {slice_by_tokens(ex['description_clean'], max_tokens=28)!r}")
print(f"  • article_clean     ({ex['n_tokens_clean']:4d} BPE tokens): {slice_by_tokens(ex['article_clean'], max_tokens=28)!r}\n")

df[["n_tokens_headline", "n_tokens_description", "n_tokens_clean", "n_tokens_search_text"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9]
).round(1)

=== Glass-Box Example: Unfielded 'search_text' (Flat BM25) vs. Separate Fielded Columns (BM25F / PassageChunker) ===
[1] Unfielded search_text (2801 BPE tokens | first 50 BPE tokens shown):
    "There's a shortage of truckers, but TuSimple thinks it has a solution: no driver needed \nThe e-commerce boom has exacerbated a global truck driver shortage, but could autonomous trucks help fix the problem? \nRight now, there's a shortage ..."

[2] Separate Fielded Inputs (preserved independently for BM25F and Contextual Chunking):
  • headline_clean    (  20 BPE tokens): "There's a shortage of truckers, but TuSimple thinks it has a solution: no driver needed"
  • description_clean (  21 BPE tokens): 'The e-commerce boom has exacerbated a global truck driver shortage, but could autonomous trucks help fix the problem?'
  • article_clean     (2758 BPE tokens): "Right now, there's a shortage of truck drivers in the US and worldwide, exacerbated by the e-commerce boom brought on by the pandemic. .

,n_tokens_headline,n_tokens_description,n_tokens_clean,n_tokens_search_text
count,37949.0,37949.0,37949.0,37949.0
mean,13.6,34.1,1274.7,1324.3
std,4.6,11.7,1453.3,1453.8
min,1.0,0.0,0.0,11.0
25%,10.0,27.0,590.0,638.0
50%,13.0,32.0,917.0,968.0
75%,16.0,40.0,1478.0,1529.0
90%,20.0,48.2,2314.2,2366.0
max,42.0,141.0,26654.0,26724.0


## 6. Persisting and Verifying the Clean Corpus

We write the enriched dataset to `cnn_english_news_clean.parquet` and run integrity assertions confirming that the row count is unchanged and all original columns match the raw file.

In [9]:
out_cols = [
    c for c in df.columns
    if c not in ("date",)
]
out = df[out_cols]

out.to_parquet(OUT_PATH, index=False)

# Verification: raw file untouched, clean file complete and bit-identical on original columns
chk = pd.read_parquet(OUT_PATH)
assert len(chk) == len(df)
assert chk["article_clean"].eq("").sum() == df["article_clean"].eq("").sum()
assert (chk["Article text"].fillna("∅") == df["Article text"].fillna("∅")).all()
print(
    f"Saved: {OUT_PATH} ({OUT_PATH.stat().st_size/1e6:.0f} MB, "
    f"{len(chk):,} rows, {chk.shape[1]} columns)"
)
print("Raw file size unchanged:", RAW_PATH.stat().st_size, "bytes")

Saved: ../data/cnn_english_news_clean.parquet (385 MB, 37,949 rows, 27 columns)
Raw file size unchanged: 133781676 bytes


## 7. Inspecting a Cleaned Record

Finally, we inspect a single processed article to see all cleaned fields and quality flags side by side.

In [10]:
from src.tokenizer import slice_by_tokens

row = chk[~chk["flag_empty_article"] & chk["flag_dup_headline"]].iloc[0]
print("headline_clean  :", row["headline_clean"], f"({row['n_tokens_headline']} BPE tokens)")
print("published       :", row["published"], f"({row['year']}-{row['month']:02d})")
print("slug            :", row["slug"])
print("flags           :", {c: bool(row[c]) for c in chk.columns if c.startswith("flag_")})
print(f"article_clean   ({row['n_tokens_clean']} BPE tokens, first 60 BPE tokens):")
print("  ", slice_by_tokens(row["article_clean"], max_tokens=60))

headline_clean  : The 10 Senate seats most likely to flip in 2022 (16 BPE tokens)
published       : 2022-02-03 11:09:19 (2022-02)
slug            : senate-race-rankings-february
flags           : {'flag_empty_article': False, 'flag_dup_headline': True, 'flag_no_date': False}
article_clean   (4311 BPE tokens, first 60 BPE tokens):
   There's nothing like a Supreme Court vacancy to get people in Washington talking about the Senate. It's less clear, however, whether the voters who will decide control of the Senate nine months from now are attuned to that conversation, especially since the ideological makeup of the court is not in the balance. ...


## Summary

We have created `cnn_english_news_clean.parquet` with 37,916 non-empty articles (`~flag_empty_article`) ready for indexing. By keeping `headline_clean`, `description_clean`, and `article_clean` as distinct columns alongside the concatenated `search_text`, we can directly compare unfielded lexical retrieval against field-weighted models and passage-level dense indexes in the following notebooks.